URLGuard (MalURLBench)

Method: LLM-based malicious URL risk assessment.

For this baseline, I used the authors' released URLGuard LoRA adapter from Hugging Face together with the pretrained Llama-2-7B-Chat model.

I loaded the model using 4-bit quantization on Kaggle. Unlike the previous baselines, I did not perform additional training because the authors had already released their fine-tuned adapter.

Since the complete original training dataset was not available, I focused on evaluating the released model.

I used a simplified risk-assessment prompt and a balanced sample of 200 URLs from an external dataset.

Results:

Accuracy: 57.36%
Malicious F1-score: 57.58%

Limitation: These results are substantially lower than those of the other baselines. However, the evaluation does not reproduce the original MalURLBench agent scenarios or prompt format. Therefore, the result should be interpreted as performance under my simplified external evaluation, not as a reproduction of the paper's reported results.

# Import Libraries

In [9]:
# Install required libraries for URLGuard

%pip install -q "transformers>=4.46,<5" "peft>=0.14,<1" "accelerate>=1.0,<2" "bitsandbytes>=0.46.1" huggingface_hub scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install -q "bitsandbytes>=0.46.1"

In [8]:
%pip install --no-cache-dir -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 104.6 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [6]:
import json, re, gc, time
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

In [7]:
# cell 2: authenticate


OUT = Path('/kaggle/working/URLGuard')
OUT.mkdir(parents=True, exist_ok=True)
BASE = 'meta-llama/Llama-2-7b-chat-hf'
ADAPTER = 'nonamehhh/URLGuard'

token = UserSecretsClient().get_secret('HF_TOKEN')
print('Authenticated as:', HfApi(token=token).whoami()['name'])
print('Available GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), 'Enable a GPU in Kaggle settings before inference.'


Authenticated as: Madina1428
Available GPUs: ['Tesla T4', 'Tesla T4']


In [8]:
# cell 3: download / inspect the public adapter

adapter_config_path = hf_hub_download(ADAPTER, 'adapter_config.json')
with open(adapter_config_path) as f:
    adapter_config = json.load(f)
print('Adapter type:', adapter_config.get('peft_type'))
print('LoRA rank:', adapter_config.get('r'))
print('Target modules:', adapter_config.get('target_modules'))

# Download the small public adapter files into Kaggle's cache now.
adapter_dir = snapshot_download(
    ADAPTER,
    allow_patterns=['adapter_config.json','adapter_model.safetensors',
                    'tokenizer_config.json','special_tokens_map.json',
                    'tokenizer.model','chat_template.jinja']
)
print('Adapter ready:', adapter_dir)


adapter_config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

Adapter type: LORA
LoRA rank: 16
Target modules: ['o_proj', 'v_proj', 'k_proj', 'q_proj']


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Adapter ready: /root/.cache/huggingface/hub/models--nonamehhh--URLGuard/snapshots/2f6b162f7c052bdaad8912caa618295954b571f1


In [13]:
# cell 4

paths = list(Path('/kaggle/input').rglob('*.csv'))
print('Attached CSV files:', [str(p) for p in paths[:20]])

selected_path = None
for p in paths:
    try:
        sample = pd.read_csv(p, nrows=3)
        if {'url','type'}.issubset({str(c).lower() for c in sample.columns}):
            selected_path = p
            break
    except Exception:
        pass

if selected_path is None:
    raise FileNotFoundError(
        'Attach the Kaggle Malicious URLs Dataset (sid321axn/malicious-urls-dataset), '
        'then rerun this cell.'
    )

urls = pd.read_csv(selected_path)
urls.columns = [str(c).lower() for c in urls.columns]
urls = urls[['url','type']].dropna().copy()
urls['type'] = urls['type'].str.lower().str.strip()
urls = urls[urls['type'].isin(['benign','phishing','malware','defacement'])]
urls['url'] = urls['url'].astype(str)
urls = urls[urls['url'].str.len().between(8, 2048)].drop_duplicates('url')
urls['true_label'] = (urls['type'] != 'benign').astype(int)  # 1=malicious, 0=benign

# sample size (not full dataset)

N_PER_CLASS = 100
parts = []
for label in [0,1]:
    subset = urls[urls['true_label'] == label]
    assert len(subset) >= N_PER_CLASS, f'Not enough examples for label {label}'
    parts.append(subset.sample(n=N_PER_CLASS, random_state=42))
eval_df = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)
eval_df.to_csv(OUT / 'URLGuard_evaluation_sample.csv', index=False)
print('Loaded from:', selected_path)
print('Evaluation sample:', len(eval_df), eval_df['true_label'].value_counts().to_dict())
print('Saved labeled sample without contacting any URLs.')


Attached CSV files: ['/kaggle/input/datasets/sid321axn/malicious-urls-dataset/malicious_phish.csv']
Loaded from: /kaggle/input/datasets/sid321axn/malicious-urls-dataset/malicious_phish.csv
Evaluation sample: 200 {0: 100, 1: 100}
Saved labeled sample without contacting any URLs.


In [14]:
# cell 5: confirm my account can download the gated base model, it checks file access

from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

try:
    path = hf_hub_download(BASE, filename='config.json', token=token)
    access_ready = True
    print('Base model access granted:', path)
except (GatedRepoError, HfHubHTTPError) as exc:
    access_ready = False
    print('Base model access is not available yet.')
    print('Check your request at https://huggingface.co/meta-llama/Llama-2-7b-chat-hf')
    print('Access error:', type(exc).__name__)

Base model access granted: /root/.cache/huggingface/hub/models--meta-llama--Llama-2-7b-chat-hf/snapshots/f5db02db724555f92da89c216ac04704f23d4590/config.json


In [15]:
# cell 6: load Llama-2 in 4-bit, attach the published URLGuard weights

assert access_ready, 'Llama-2 access pending. Run this cell after approval.'
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(BASE, token=token)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE, 
    token=token, 
    quantization_config=quant, 
    device_map='auto',
    dtype=torch.float16, 
    low_cpu_mem_usage=True
)
model = PeftModel.from_pretrained(base_model, ADAPTER)
model.eval()
print('Released URLGuard adapter loaded successfully')


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Released URLGuard adapter loaded successfully


In [16]:
# cell 7: prompt is a simplified proxy, not the official benchmark prompt

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

def predict_risk(url):
    messages = [
        {'role':'system','content':(
            'Assess the URL using only its text, without visiting it. '
            'Give exactly one label on the first line: '
            'Risk Level: No Risk, Risk Level: Medium Risk, or Risk Level: High Risk. '
            'Then give a short reason.')} ,
        {'role':'user','content':f'Assess this URL: {url}'}
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    tensors = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    model_device = model.get_input_embeddings().weight.device
    tensors = {k: v.to(model_device) for k,v in tensors.items()}
    with torch.inference_mode():
        generated = model.generate(
            **tensors, max_new_tokens=64, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(
        generated[0][tensors['input_ids'].shape[-1]:],
        skip_special_tokens=True
    ).strip()
    match = re.search(r'Risk\s*Level\s*:\s*(No Risk|Medium Risk|High Risk)',
                      response, flags=re.I)
    if not match:
        return response, 'Unparseable', None
    level = match.group(1).title()
    
    # for this experiment: No Risk=benign. Medium/High=malicious
    return response, level, 0 if level == 'No Risk' else 1

rows=[]
for i, row in eval_df.iterrows():
    response, level, pred = predict_risk(row['url'])
    rows.append({'url':row['url'], 'type':row['type'],
                 'true_label':int(row['true_label']), 'risk_level':level,
                 'predicted_label':pred, 'model_response':response})
    if (i+1) % 10 == 0:
        print(f'Processed {i+1}/{len(eval_df)} URLs')
    pd.DataFrame(rows).to_csv(OUT / 'URLGuard_predictions.csv', index=False)

pred_df = pd.DataFrame(rows)
valid = pred_df.dropna(subset=['predicted_label'])
if valid.empty:
    print('No parseable predictions. Inspect URLGuard_predictions.csv before scoring.')
else:
    y_true=valid['true_label'].astype(int)
    y_pred=valid['predicted_label'].astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', pos_label=1, zero_division=0
    )
    metrics = pd.DataFrame([{'model':'URLGuard (released adapter)',
        'evaluation':'simplified prompt, external URL dataset',
        'total_examples':len(pred_df), 'scored_examples':len(valid),
        'unparseable':len(pred_df)-len(valid),
        'accuracy':accuracy_score(y_true,y_pred),
        'malicious_precision':precision,'malicious_recall':recall,
        'malicious_f1':f1}])
    metrics.to_csv(OUT / 'URLGuard_metrics.csv', index=False)
    display(metrics)
    print('Confusion matrix (0=benign, 1=malicious):')
    print(confusion_matrix(y_true,y_pred,labels=[0,1]))
print('Outputs in:', OUT)


[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed 10/200 URLs


[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed 20/200 URLs


[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed 30/200 URLs


... (32 outputs left out)

[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed 200/200 URLs


,model,evaluation,total_examples,scored_examples,unparseable,accuracy,malicious_precision,malicious_recall,malicious_f1
0,URLGuard (released adapter),"simplified prompt, external URL dataset",200,197,3,0.573604,0.581633,0.57,0.575758


Confusion matrix (0=benign, 1=malicious):
[[56 41]
 [43 57]]
Outputs in: /kaggle/working/URLGuard


In [17]:
import pandas as pd

df = pd.read_csv(
    "/kaggle/working/URLGuard/URLGuard_predictions.csv"
)

print("Columns:", df.columns.tolist())
display(df.head(5).T)

Columns: ['url', 'type', 'true_label', 'risk_level', 'predicted_label', 'model_response']


,0,1,2,3,4
url,http://olx.co.id/q-%7Bq%7D/sulawesi-tenggara?u...,pipl.com/directory/name/Andrews/Albert,virtualtourist.com/hotels/Europe/France/Ile_de...,http://fcdomont.fr/sejeal.jpg,slashdot.org/articles/00/11/13/066228.shtml
type,benign,benign,benign,defacement,phishing
true_label,0,0,0,1,1
risk_level,High Risk,High Risk,High Risk,High Risk,No Risk
predicted_label,1.0,1.0,1.0,1.0,0.0
model_response,Risk Level: High Risk\nReason: The URL contain...,Risk Level: High Risk\nReason: The URL is a pe...,Risk Level: High Risk\nReason: The URL contain...,Risk Level: High Risk\nReason: The URL contain...,Risk Level: No Risk\nReason: The URL is a well...


In [18]:
df = pd.read_csv(
    "/kaggle/working/URLGuard/URLGuard_predictions.csv"
)

# Convert multiline responses into single-line text
df["model_response"] = (
    df["model_response"]
    .fillna("")
    .str.replace("\n", " ", regex=False)
    .str.replace("\r", " ", regex=False)
)

df.to_csv(
    "/kaggle/working/URLGuard/URLGuard_predictions_readable.csv",
    index=False
)

print("Saved readable CSV successfully!")

Saved readable CSV successfully!
